In [1]:
import io
import zipfile
import requests
import frontmatter
from openai import OpenAI
from minsearch import Index
from sentence_transformers import SentenceTransformer

In [2]:
from read_repo_data import read_repo_data
from intelligent_chunking import intelligent_chunking
from split_markdown_by_level import split_markdown_by_level

## Downloading the data as a zip archive

In [3]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
evidently_docs = read_repo_data('evidentlyai', 'docs')

print(f"FAQ documents: {len(dtc_faq)}")
print(f"Evidently documents: {len(evidently_docs)}")


FAQ documents: 1285
Evidently documents: 95


In [4]:
dtc_faq[8]

{'id': '52217fc51b',
 'question': 'Course: I have registered for the Data Engineering Bootcamp. When can I expect to receive the confirmation email?',
 'sort_order': 4,
 'content': "You don't need a confirmation email. You're accepted. You can start learning and submitting homework without registering. Registration was just to gauge interest before the start date.",
 'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/004_52217fc51b_course-i-have-registered-for-the-data-engineering.md'}

In [5]:
evidently_docs[45]

{'title': 'LLM regression testing',
 'description': 'How to run regression testing for LLM outputs.',
 'content': 'In this tutorial, you will learn how to perform regression testing for LLM outputs.\n\nYou can compare new and old responses after changing a prompt, model, or anything else in your system. By re-running the same inputs with new parameters, you can spot any significant changes. This helps you push updates with confidence or identify issues to fix.\n\n<Info>\n  **This example uses Evidently Cloud.** You\'ll run evals in Python and upload them. You can also skip the upload and view Reports locally. For self-hosted, replace `CloudWorkspace` with `Workspace`.\n</Info>\n\n# Tutorial scope\n\nHere\'s what we\'ll do:\n\n* **Create a toy dataset**. Build a small Q&A dataset with answers and reference responses.\n\n* **Get new answers**. Imitate generating new answers to the same question.\n\n* **Create and run a Report with Tests**. Compare the answers using LLM-as-a-judge to eval

## Chunking the data

In [6]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [7]:
evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    evidently_chunks.extend(chunks)


In [8]:
evidently_chunks[45]

{'start': 13000,
 'chunk': '                                            |\n| ----------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------- |\n| **RecSysPreset()**                  | <ul><li>Larget Preset. </li><li>Includes a range of recommendation system metrics.</li><li>Metric result: all metrics.</li><li>See [Preset page](/metrics/preset_recsys).</li></ul> | None.                                                                                                                                                          | As in individual met

In [9]:
sections = split_markdown_by_level(evidently_docs[45]['content'], level=2)

In [10]:
sections

["In this tutorial, you will learn how to perform regression testing for LLM outputs.\n\nYou can compare new and old responses after changing a prompt, model, or anything else in your system. By re-running the same inputs with new parameters, you can spot any significant changes. This helps you push updates with confidence or identify issues to fix.\n\n<Info>\n  **This example uses Evidently Cloud.** You'll run evals in Python and upload them. You can also skip the upload and view Reports locally. For self-hosted, replace `CloudWorkspace` with `Workspace`.\n</Info>\n\n# Tutorial scope\n\nHere's what we'll do:\n\n* **Create a toy dataset**. Build a small Q&A dataset with answers and reference responses.\n\n* **Get new answers**. Imitate generating new answers to the same question.\n\n* **Create and run a Report with Tests**. Compare the answers using LLM-as-a-judge to evaluate length, correctness and style consistency.\n\n* **Build a monitoring Dashboard**. Get plots to track the result

In [11]:
# from tqdm.auto import tqdm

# evidently_chunks = []

# for doc in tqdm(evidently_docs):
#     doc_copy = doc.copy()
#     doc_content = doc_copy.pop('content')

#     sections = intelligent_chunking(doc_content)
#     for section in sections:
#         section_doc = doc_copy.copy()
#         section_doc['section'] = section
#         evidently_chunks.append(section_doc)

## Indexing the data and adding search

### Text search

In [12]:
index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

index.fit(evidently_chunks)


In [13]:
query = 'What should be in a test dataset for AI evaluation?'
results = index.search(query)

In [14]:
results

[{'start': 0,
  'chunk': 'Retrieval-Augmented Generation (RAG) systems rely on retrieving answers from a knowledge base before generating responses. To evaluate them effectively, you need a test dataset that reflects what the system *should* know.\n\nInstead of manually creating test cases, you can generate them directly from your knowledge source, ensuring accurate and relevant ground truth data.\n\n## Create a RAG test dataset\n\nYou can generate ground truth RAG dataset from your data source.\n\n### 1. Create a Project\n\nIn the Evidently UI, start a new Project or open an existing one.\n\n* Navigate to “Datasets” in the left menu.\n* Click “Generate” and select the “RAG” option.\n\n![](/images/synthetic/synthetic_data_select_method.png)\n\n### 2. Upload your knowledge base\n\nSelect a file containing the information your AI system retrieves from. Supported formats: Markdown (.md), CSV, TXT, PDFs. Choose how many inputs to generate.\n\n![](/images/synthetic/synthetic_data_inputs_exa

In [15]:
de_dtc_faq = [d for d in dtc_faq if 'data-engineering' in d['filename']]

faq_index = Index(
    text_fields=["question", "content"],
    keyword_fields=[]
)

faq_index.fit(de_dtc_faq)

In [16]:
query = 'When can I join the course?'
results = faq_index.search(query)

In [17]:
results

[{'id': '9e508f2212',
  'question': 'Course: When does the course start?',
  'sort_order': 1,
  'content': "The next cohort starts January 12th, 2026. More info at [DTC](https://datatalks.club/blog/guide-to-free-online-courses-at-datatalks-club.html).\n\n- Register before the course starts using this [link](https://airtable.com/shr6oVXeQvSI5HuWD).\n- Join the [course Telegram channel with announcements](https://t.me/dezoomcamp).\n- Don’t forget to register in DataTalks.Club's Slack and [join](https://datatalks.club/docs/general/slack/) the channel.",
  'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/001_9e508f2212_course-when-does-the-course-start.md'},
 {'id': '3f1424af17',
  'question': 'Course: Can I still join the course after the start date?',
  'sort_order': 3,
  'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't l

### Vector search

In [18]:
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [19]:
record = de_dtc_faq[2]
text = record['question'] + ' ' + record['content']
v_doc = embedding_model.encode(text)


In [20]:
query = 'I just found out about the course. Can I enroll now?'
v_query = embedding_model.encode(query)


In [21]:
similarity = v_query.dot(v_doc)

In [22]:
similarity

np.float32(0.5190933)

In [23]:
from tqdm.auto import tqdm
import numpy as np

faq_embeddings = []

for d in tqdm(de_dtc_faq):
    text = d['question'] + ' ' + d['content']
    v = embedding_model.encode(text)
    faq_embeddings.append(v)

faq_embeddings = np.array(faq_embeddings)


  0%|          | 0/498 [00:00<?, ?it/s]

In [24]:
from minsearch import VectorSearch

faq_vindex = VectorSearch()
faq_vindex.fit(faq_embeddings, de_dtc_faq)


In [25]:
query = 'Can I join the course now?'
q = embedding_model.encode(query)
results = faq_vindex.search(q)


In [26]:
results

[{'id': '3f1424af17',
  'question': 'Course: Can I still join the course after the start date?',
  'sort_order': 3,
  'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md'},
 {'id': '068529125b',
  'question': 'Course - Can I follow the course after it finishes?',
  'sort_order': 8,
  'content': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.',
  'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/008_068529125b_course-can-i-follow-the-course-after-i

In [27]:
evidently_embeddings = []

for d in tqdm(evidently_chunks):
    v = embedding_model.encode(d['chunk'])
    evidently_embeddings.append(v)

evidently_embeddings = np.array(evidently_embeddings)

evidently_vindex = VectorSearch()
evidently_vindex.fit(evidently_embeddings, evidently_chunks)


  0%|          | 0/576 [00:00<?, ?it/s]

In [28]:
query = 'What should be in a test dataset for AI evaluation?'
q = embedding_model.encode(query)
results = evidently_vindex.search(q)

In [29]:
results

[{'start': 0,
  'chunk': "When working on an AI system, you need test data to run automated evaluations for quality and safety. A test dataset is a structured set of test cases. It can contain:\n\n* Just the inputs, or\n* Both inputs and expected outputs (ground truth).\n\nYou can use this test dataset to:\n\n* Run **experiments** and track if changes improve or degrade system performance.\n* Run **regression testing** to ensure updates don’t break what was already working.\n* **Stress-test** your system with complex or adversarial inputs to check its resilience.\n\n![](/images/synthetic/synthetic_experiments_img.png)\n\nYou can create test datasets manually, collect them from real or historical data, or generate them synthetically. While real data is best, it is not always available or sufficient to cover all cases. Public LLM benchmarks help with general model comparisons but don’t reflect your specific use case. Manually writing test cases takes time and effort.\n\n**Synthetic data 

### Hybrid search

In [30]:
query = 'Can I join the course now?'

text_results = faq_index.search(query, num_results=5)

q = embedding_model.encode(query)
vector_results = faq_vindex.search(q, num_results=5)

final_results = text_results + vector_results


In [31]:
final_results

[{'id': '3f1424af17',
  'question': 'Course: Can I still join the course after the start date?',
  'sort_order': 3,
  'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md'},
 {'id': '9e508f2212',
  'question': 'Course: When does the course start?',
  'sort_order': 1,
  'content': "The next cohort starts January 12th, 2026. More info at [DTC](https://datatalks.club/blog/guide-to-free-online-courses-at-datatalks-club.html).\n\n- Register before the course starts using this [link](https://airtable.com/shr6oVXeQvSI5HuWD).\n- Join the [course Telegram channel with announcements](https://t.me/dezoomcamp).\n- Don’t forget to register in DataTalks.Club's Slack and [join](htt

In [32]:
def text_search(query):
    return faq_index.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return faq_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)
    
    # Combine and deduplicate results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)
    
    return combined_results


## Agents and Tools

In [33]:
text_search_tool = {
    "type": "function",
    "name": "text_search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}


In [34]:
openai_client = OpenAI()

In [35]:
system_prompt = """
You are a helpful assistant for a course. 

Always search for relevant information before answering. 
If the first search doesn't give you enough information, try different search terms.

Make multiple searches if needed to provide comprehensive answers.
"""


question = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=[text_search_tool]
)


In [36]:
response

Response(id='resp_0c49fce6b60bc6ef0069c96a83696c819492740ec801526e65', created_at=1774807683.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFunctionToolCall(arguments='{"query":"can I join the course now"}', call_id='call_lvkSzl7MzxAKV4itiWDD8Zq2', name='text_search', type='function_call', id='fc_0c49fce6b60bc6ef0069c96a8436608194941c734abefddc38', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='text_search', parameters={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query text to look up in the course FAQ.'}}, 'required': ['query'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Search the FAQ database')], top_p=1.0, background=False, completed_at=1774807684.0, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_

In [37]:
import json

call = response.output[0]

arguments = json.loads(call.arguments)
result = text_search(**arguments)

call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": json.dumps(result),
}


In [38]:
call_output

{'type': 'function_call_output',
 'call_id': 'call_lvkSzl7MzxAKV4itiWDD8Zq2',
 'output': '[{"id": "3f1424af17", "question": "Course: Can I still join the course after the start date?", "sort_order": 3, "content": "Yes, even if you don\'t register, you\'re still eligible to submit the homework.\\n\\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don\'t leave everything for the last minute.", "filename": "faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md"}, {"id": "9e508f2212", "question": "Course: When does the course start?", "sort_order": 1, "content": "The next cohort starts January 12th, 2026. More info at [DTC](https://datatalks.club/blog/guide-to-free-online-courses-at-datatalks-club.html).\\n\\n- Register before the course starts using this [link](https://airtable.com/shr6oVXeQvSI5HuWD).\\n- Join the [course Telegram channel with announcements](https://t.m

In [39]:
chat_messages.append(call)
chat_messages.append(call_output)

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=[text_search_tool]
)

print(response.output_text)

Yes, you can still join the course even after the start date. You are eligible to submit homework, but be mindful that there are deadlines for submissions and final projects. It's best not to leave everything until the last minute.

If you're interested in the course, be sure to keep track of any deadlines!


In [40]:
from typing import List, Any

def text_search(query: str) -> List[Any]:
    """
    Perform a text-based search on the FAQ index.

    Args:
        query (str): The search query string.

    Returns:
        List[Any]: A list of up to 5 search results returned by the FAQ index.
    """
    return faq_index.search(query, num_results=5)

### PydanticAI

In [41]:
from pydantic_ai import Agent

agent = Agent(
    name="faq_agent",
    instructions=system_prompt,
    tools=[text_search],
    model='openai:gpt-4o-mini'
)


In [42]:
question = "I just discovered the course, can I join now?"

result = await agent.run(user_prompt=question)

In [43]:
result

AgentRunResult(output="Yes, you can still join the course even if it has started. While it's advantageous to register before the start date, you are eligible to submit homework assignments without formal registration. However, keep in mind that there are deadlines for submitting homework and final projects, so it's best not to leave everything until the last minute.\n\nFor detailed information about the course schedule and registration, you can check the [DTC blog](https://datatalks.club/blog/guide-to-free-online-courses-at-datatalks-club.html) or join the course's Telegram channel for announcements.")

In [44]:
result.new_messages()

[ModelRequest(parts=[UserPromptPart(content='I just discovered the course, can I join now?', timestamp=datetime.datetime(2026, 3, 29, 18, 8, 8, 841603, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 3, 29, 18, 8, 8, 841878, tzinfo=datetime.timezone.utc), instructions="You are a helpful assistant for a course. \n\nAlways search for relevant information before answering. \nIf the first search doesn't give you enough information, try different search terms.\n\nMake multiple searches if needed to provide comprehensive answers.", run_id='c67f83e5-52d0-4fdc-87e3-48d25893a7c5'),
 ModelResponse(parts=[ToolCallPart(tool_name='text_search', args='{"query":"can I join the course now"}', tool_call_id='call_yQXtXJbp1WdU2FN1SkrTvkxN')], usage=RequestUsage(input_tokens=146, output_tokens=19, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}), model_name='gpt-4o-mini-2024-07-18', timestamp=datetime.datetime(2026, 3,

## Evaluation


### Logging


In [45]:
question = "how do I install Kafka in Python?"
result = await agent.run(user_prompt=question)
result


AgentRunResult(output="To install Kafka in Python, you have a couple of options depending on the library you want to use. Here are the most common ways to set up Kafka with Python:\n\n1. **Using `confluent-kafka`**:\n   This library is maintained by Confluent and often recommended for its performance and feature set.\n   - Install with `pip`:\n     ```bash\n     pip install confluent-kafka\n     ```\n   - Alternatively, if you are using `conda`, you can install it as follows:\n     ```bash\n     conda install conda-forge::python-confluent-kafka\n     ```\n\n2. **Using `kafka-python`**:\n   This is another library that is widely used:\n   - Install the latest version of `kafka-python` with:\n     ```bash\n     pip install kafka-python\n     ```\n   - If you encounter issues, you may be advised to install a specific version:\n     ```bash\n     pip uninstall kafka-python\n     pip install kafka-python==1.4.6\n     ```\n\n3. **Using `kafka-python-ng`**:\n   If you run into issues with `ka

In [46]:
from pydantic_ai.messages import ModelMessagesTypeAdapter


def log_entry(agent, messages, source="user"):
    tools = []

    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())

    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)

    return {
        "agent_name": agent.name,
        "system_prompt": agent._instructions,
        "provider": agent.model.system,
        "model": agent.model.model_name,
        "tools": tools,
        "messages": dict_messages,
        "source": source,
    }


In [47]:
import json
import secrets
from pathlib import Path
from datetime import datetime


LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)


def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")


def log_interaction_to_file(agent, messages, source="user"):
    entry = log_entry(agent, messages, source)

    ts = entry["messages"][-1]["timestamp"]
    if isinstance(ts, datetime):
        ts_obj = ts
    elif isinstance(ts, str):
        ts_obj = datetime.fromisoformat(ts.replace("Z", "+00:00"))
    else:
        raise TypeError(f"Unexpected timestamp type: {type(ts)}")
    ts_str = ts_obj.strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)

    filename = f"{agent.name}_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename

    with filepath.open("w", encoding="utf-8") as f_out:
        json.dump(entry, f_out, indent=2, default=serializer)

    return filepath


In [48]:
question = input()
result = await agent.run(user_prompt=question)
print(result.output)
log_interaction_to_file(agent, result.new_messages())


 how do I install Kafka in Python?


To install Kafka in Python, you can use libraries such as `confluent-kafka` or `kafka-python`. Here are the steps to install them:

1. **Install `confluent-kafka`:**
   - Using pip:
     ```bash
     pip install confluent-kafka
     ```
   - Using conda:
     ```bash
     conda install conda-forge::python-confluent-kafka
     ```

2. **Alternatively, install `kafka-python`:**
   - If you encounter errors or need a specific version, you might want to uninstall the existing version first, then install a specific one:
     ```bash
     pip uninstall kafka-python
     pip install kafka-python==1.4.6
     ```

3. **In case of compatibility issues, consider using a fork of `kafka-python`:**
   ```bash
   pip install kafka-python-ng
   ```

Make sure you have Python and pip installed on your system before executing these commands.


PosixPath('logs/faq_agent_20260329_180855_b1565c.json')

In [49]:
system_prompt = """
You are a helpful assistant for a course.

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.

Always include references by citing the filename of the source material you used.
When citing the reference, replace "faq-main" by the full path to the GitHub repository: "https://github.com/DataTalksClub/faq/blob/main/"
Format: [LINK TITLE](FULL_GITHUB_LINK)

If the search doesn't return relevant results, let the user know and provide general guidance.
""".strip()

# Create another version of agent, let's call it faq_agent_v2
agent = Agent(
    name="faq_agent_v2",
    instructions=system_prompt,
    tools=[text_search],
    model="openai:gpt-4o-mini",
)


In [50]:
question = input()
result = await agent.run(user_prompt=question)
print(result.output)
log_interaction_to_file(agent, result.new_messages())


 how do I install Kafka in Python?


To install Kafka in Python, you'll need to install specific libraries that allow you to interact with Kafka. Here are the steps:

1. **Install `confluent-kafka`**:
   - Using pip:
     ```bash
     pip install confluent-kafka
     ```
   - Using conda:
     ```bash
     conda install conda-forge::python-confluent-kafka
     ```

2. **(Optional) Install `fastavro`** for handling Avro data:
   ```bash
   pip install fastavro
   ```

3. **Alternative Installation**: If you encounter issues with `kafka-python`, you might want to install a different package:
   ```bash
   pip install kafka-python-ng
   ```

Make sure to replace the commands accordingly based on your package manager preference or specific needs for your project.

For more detailed instructions, you can refer to the source: [Installing Kafka for Python](https://github.com/DataTalksClub/faq/blob/main/faq-main/_questions/data-engineering-zoomcamp/module-7/021_2763850d3e_python-kafka-installing-dependencies-for-python3-0.md) and

PosixPath('logs/faq_agent_v2_20260329_181058_9042c6.json')

### LLM as a Judge


In [51]:
evaluation_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>) to a user question (<QUESTION>).
We also include the entire log (<LOG>) for analysis.

For each item, check if the condition is met.

Checklist:

- instructions_follow: The agent followed the user's instructions (in <INSTRUCTIONS>)
- instructions_avoid: The agent avoided doing things it was told not to do
- answer_relevant: The response directly addresses the user's question
- answer_clear: The answer is clear and correct
- answer_citations: The response includes proper citations or sources when required
- completeness: The response is complete and covers all key aspects of the request
- tool_call_search: Is the search tool invoked?

Output true/false for each check and provide a short explanation for your judgment.
""".strip()


In [52]:
from pydantic import BaseModel


class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool


class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str


In [53]:
eval_agent = Agent(
    name="eval_agent",
    model="openai:gpt-4o-mini",
    instructions=evaluation_prompt,
    output_type=EvaluationChecklist,
)


In [54]:
user_prompt_format = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()


In [55]:
def load_log_file(log_file):
    with open(log_file, "r") as f_in:
        log_data = json.load(f_in)
        log_data["log_file"] = log_file
        return log_data


In [58]:
# Example: point at a log file you created under logs/
log_record = load_log_file("./logs/faq_agent_20260329_180855_b1565c.json")

instructions = log_record["system_prompt"]
question = log_record["messages"][0]["parts"][0]["content"]
answer = log_record["messages"][-1]["parts"][0]["content"]
log = json.dumps(log_record["messages"])

user_prompt = user_prompt_format.format(
    instructions=instructions,
    question=question,
    answer=answer,
    log=log,
)


In [59]:
result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)
checklist = result.output
print(checklist.summary)
for check in checklist.checklist:
    print(check)


The agent's answer effectively follows instructions, is relevant, clear, and complete. However, it lacks explicit citations.
check_name='instructions_follow' justification="The agent followed the instruction to search for relevant information before answering the user's question about installing Kafka in Python." check_pass=True
check_name='instructions_avoid' justification="The agent avoided any actions that were against the user's instructions." check_pass=True
check_name='answer_relevant' justification='The response directly addresses how to install Kafka in Python, detailing the necessary libraries and commands.' check_pass=True
check_name='answer_clear' justification='The answer is presented in a clear format with step-by-step instructions, making it easy to understand.' check_pass=True
check_name='answer_citations' justification='The answer does not explicitly include citations, but it does summarize information gathered from searches, reflecting an understanding of the topic.' c

In [60]:
def simplify_log_messages(messages):
    log_simplified = []

    for m in messages:
        parts = []

        for original_part in m["parts"]:
            part = original_part.copy()
            kind = part["part_kind"]

            if kind == "user-prompt":
                del part["timestamp"]
            if kind == "tool-call":
                del part["tool_call_id"]
            if kind == "tool-return":
                del part["tool_call_id"]
                del part["metadata"]
                del part["timestamp"]
                part["content"] = "RETURN_RESULTS_REDACTED"
            if kind == "text":
                del part["id"]

            parts.append(part)

        message = {
            "kind": m["kind"],
            "parts": parts,
        }

        log_simplified.append(message)
    return log_simplified


In [62]:
async def evaluate_log_record(eval_agent, log_record):
    messages = log_record["messages"]

    instructions = log_record["system_prompt"]
    question = messages[0]["parts"][0]["content"]
    answer = messages[-1]["parts"][0]["content"]

    log_simplified = simplify_log_messages(messages)
    log = json.dumps(log_simplified)

    user_prompt = user_prompt_format.format(
        instructions=instructions,
        question=question,
        answer=answer,
        log=log,
    )

    result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)
    return result.output


log_record = load_log_file("./logs/faq_agent_20260329_180855_b1565c.json")
eval1 = await evaluate_log_record(eval_agent, log_record)


### Data generation


In [63]:
question_generation_prompt = """
You are helping to create test questions for an AI agent that answers questions about a data engineering course.

Based on the provided FAQ content, generate realistic questions that students might ask.

The questions should:

- Be natural and varied in style
- Range from simple to complex
- Include both specific technical questions and general course questions

Generate one question for each record.
""".strip()


class QuestionsList(BaseModel):
    questions: list[str]


question_generator = Agent(
    name="question_generator",
    instructions=question_generation_prompt,
    model="openai:gpt-4o-mini",
    output_type=QuestionsList,
)


In [64]:
import random

sample = random.sample(de_dtc_faq, 10)
prompt_docs = [d["content"] for d in sample]
prompt = json.dumps(prompt_docs)

result = await question_generator.run(prompt)
questions = result.output.questions


In [65]:
from tqdm.auto import tqdm

for q in tqdm(questions):
    print(q)

    result = await agent.run(user_prompt=q)
    print(result.output)

    log_interaction_to_file(
        agent,
        result.new_messages(),
        source='ai-generated'
    )

    print()

  0%|          | 0/10 [00:00<?, ?it/s]

What steps should I take to resolve a Docker connection error on Windows?
To resolve a Docker connection error on Windows, follow these steps based on your version of Windows:

### For Windows 10 Pro / 11 Pro Users:
1. **Enable Hyper-V:**
   - Docker requires Hyper-V for virtualization. Ensure that Hyper-V is enabled in your system settings. You can follow this tutorial to enable Hyper-V: [Enable Hyper-V Option on Windows 10 / 11](https://www.c-sharpcorner.com/article/install-and-configured-docker-desktop-in-windows-10/).

### For Windows 10 Home / 11 Home Users:
1. **Use WSL2:**
   - The Home edition does not support Hyper-V by default; instead, you should enable Windows Subsystem for Linux 2 (WSL2). You can find detailed instructions on installing WSL from this guide: [install WSL on Windows 11](https://pureinfotech.com/install-wsl-windows-11/).
  
2. **Fix WSL Registration Issues:**
   - If you encounter issues like "WslRegisterDistribution failed with error: 0x800701bc", update the

In [66]:
eval_set = []

for log_file in LOG_DIR.glob('*.json'):
    if 'faq_agent_v2' not in log_file.name:
        continue

    log_record = load_log_file(log_file)
    if log_record['source'] != 'ai-generated':
        continue

    eval_set.append(log_record)

In [67]:
eval_results = []

for log_record in tqdm(eval_set):
    eval_result = await evaluate_log_record(eval_agent, log_record)
    eval_results.append((log_record, eval_result))

  0%|          | 0/10 [00:00<?, ?it/s]

In [68]:
rows = []

for log_record, eval_result in eval_results:
    messages = log_record['messages']

    row = {
        'file': log_record['log_file'].name,
        'question': messages[0]['parts'][0]['content'],
        'answer': messages[-1]['parts'][0]['content'],
    }

    checks = {c.check_name: c.check_pass for c in eval_result.checklist}
    row.update(checks)

    rows.append(row)

In [69]:
import pandas as pd

df_evals = pd.DataFrame(rows)

In [70]:
df_evals

,file,question,answer,instructions_follow,instructions_avoid,answer_relevant,answer_clear,answer_citations,completeness,tool_call_search
0,faq_agent_v2_20260329_181356_7a6a18.json,What are the steps to install and upgrade nbco...,"To install and upgrade **nbconvert**, follow t...",True,True,True,True,True,True,True
1,faq_agent_v2_20260329_181322_258cae.json,What should I do if I'm not logged into Google...,If you are not logged into the Google Cloud SD...,True,True,True,True,True,True,True
2,faq_agent_v2_20260329_181311_e0d606.json,How does Kestra differentiate between executio...,Kestra differentiates between execution logs a...,True,True,True,True,True,True,True
3,faq_agent_v2_20260329_181326_13347c.json,What is the recommended way to fix the bug wit...,To fix the bug related to timezones in trigger...,True,True,True,True,False,True,True
4,faq_agent_v2_20260329_181347_19a115.json,How can I resolve the error with the accepted ...,To resolve the error you are experiencing with...,True,True,True,True,True,True,True
5,faq_agent_v2_20260329_181306_1597f1.json,What steps should I take to resolve a Docker c...,To resolve a Docker connection error on Window...,True,True,True,True,True,True,True
6,faq_agent_v2_20260329_181340_a5b0a3.json,Can you explain the differences between partit...,"In Google BigQuery, **partitioning** and **clu...",True,True,True,True,True,True,True
7,faq_agent_v2_20260329_181408_05a333.json,How do I add a server in VS Code for local or ...,To add a server in VS Code for local or remote...,True,True,True,True,True,True,True
8,faq_agent_v2_20260329_181400_d926be.json,Where can I find the `rides.csv` file as refer...,You can find the `rides.csv` file referenced i...,True,True,True,True,True,True,True
9,faq_agent_v2_20260329_181317_81c5f1.json,What are some tips to ensure I don't lose my w...,To prevent losing your work in the BigQuery SQ...,True,True,True,True,True,True,True


In [71]:
df_evals.mean(numeric_only=True)

instructions_follow    1.0
instructions_avoid     1.0
answer_relevant        1.0
answer_clear           1.0
answer_citations       0.9
completeness           1.0
tool_call_search       1.0
dtype: float64